# Transformar Datos Employees - String a JSON
- Prepocesar la cadena JSON para corregir los problemas de calidad de datos
- Transformar la cadena JSON en un objeto JSON
- Escribir los datos transformados en el schema silver employees_json

In [0]:
SELECT 
*
FROM zenviro.bronze.v_employees limit 10;

In [0]:
SELECT 
value,
regexp_replace(value,'"birth_date": (\\d{4}-\\d{2}-\\d{2})','"birth_date: "\$1"') AS fixed_value
FROM zenviro.bronze.v_employees;

In [0]:
CREATE OR REPLACE TEMPORARY VIEW tv_employees
AS
SELECT 
value,
regexp_replace(value,'"birth_date": (\\d{4}-\\d{2}-\\d{2})','"birth_date: "\$1"') AS fixed_value
FROM zenviro.bronze.v_employees;

In [0]:
SELECT 
schema_of_json(fixed_value) AS schema
FROM tv_employees limit 1;

In [0]:
SELECT 
from_json(fixed_value,
        'STRUCT<birth_date: STRING, department_id: BIGINT, email: STRING, employee_id: BIGINT, gender: STRING, hire_date: STRING, job_position_id: BIGINT, name: ARRAY<STRUCT<first_name: STRING, last_name: STRING>>, phone: STRING, status: BIGINT>') AS json_value
FROM tv_employees;

In [0]:
CREATE TABLE IF NOT EXISTS zenviro.silver.employees_json
AS
SELECT 
from_json(fixed_value,
        'STRUCT<birth_date: STRING, department_id: BIGINT, email: STRING, employee_id: BIGINT, gender: STRING, hire_date: STRING, job_position_id: BIGINT, name: ARRAY<STRUCT<first_name: STRING, last_name: STRING>>, phone: STRING, status: BIGINT>') AS json_value
FROM tv_employees;

In [0]:
SELECT *FROM zenviro.silver.employees_json